In [2]:
import os
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = os.environ['PATH'] + r';C:\hadoop\bin'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, isnull, avg, sum

spark = SparkSession.builder \
    .appName("Handling Nulls") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
bookings_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_10\booking.csv",
    header=True,
    inferSchema=True
)
bookings_df.show(truncate=False)
bookings_df.printSchema()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [4]:
# 1
bookings_df.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [5]:
# 2
bookings_df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)



In [6]:
# 3
print(bookings_df.count())

15


In [7]:
# 4
bookings_df.filter(col('city').isNull()).show(truncate=False)

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider      |booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|1003      |John Mathew  |NULL|Flight      |Air India     |12000         |Confirmed     |UPI         |
|1011      |Farhan Ali   |NULL|Hotel       |Skyline Suites|22000         |Confirmed     |NULL        |
+----------+-------------+----+------------+--------------+--------------+--------------+------------+



In [8]:
# 5
bookings_df.filter(col('provider').isNull()).show(truncate=False)

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL    |7500          |Pending       |Cash        |
|1013      |Arjun Verma  |Chennai  |Flight      |NULL    |15000         |Cancelled     |Cash        |
+----------+-------------+---------+------------+--------+--------------+--------------+------------+



In [9]:
# 6
bookings_df.filter(col('booking_amount').isNull()).show(truncate=False)

+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|booking_id|customer_name|city  |service_type|provider     |booking_amount|booking_status|payment_mode|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|1005      |Vikram Rao   |Mumbai|Flight      |Vistara      |NULL          |Confirmed     |Card        |
|1014      |Kavya Nair   |Mumbai|Hotel       |Sea View Stay|NULL          |Pending       |Card        |
+----------+-------------+------+------------+-------------+--------------+--------------+------------+



In [10]:
# 7
bookings_df.filter(col('booking_status').isNull()).show(truncate=False)

+----------+-------------+----+------------+----------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider  |booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+----------+--------------+--------------+------------+
|1007      |Imran Ali    |Pune|Hotel       |Budget Inn|2200          |NULL          |UPI         |
+----------+-------------+----+------------+----------+--------------+--------------+------------+



In [11]:
# 8
bookings_df.filter(col('payment_mode').isNull()).show(truncate=False)

+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city |service_type|provider      |booking_amount|booking_status|payment_mode|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|1006      |Divya Sharma |Delhi|Flight      |IndiGo        |5900          |Cancelled     |NULL        |
|1011      |Farhan Ali   |NULL |Hotel       |Skyline Suites|22000         |Confirmed     |NULL        |
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+



In [12]:
# 9
from pyspark.sql.functions import col, count, when
null_counts = bookings_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bookings_df.columns
])
null_counts.show()

+----------+-------------+----+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------+--------------+--------------+------------+
|         0|            0|   2|           1|       2|             2|             1|           2|
+----------+-------------+----+------------+--------+--------------+--------------+------------+



In [13]:
# 10
bookings_df.dropna().show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1008      |Meera Nair   |Kochi    |Hotel       |Hill View Resort|7500          |Confirmed     |Card        |
|1009      |Rohan Das    |Kolkata  |Flight      |Air India       |7400          |Pending       |UPI         |
|1010      |Nisha Reddy  |Bangalore|Flight      |British Airways |62000         |Confirmed     |Card        |
|1015      |Ravi Kumar   |Delhi    |Flight      |SpiceJet        |4800          |Confirmed     |UPI         |
+---------

In [14]:
# 11
bookings_df.dropna(subset=['booking_amount']).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007      |Imran Ali    |Pune     |Hotel       |Budget Inn      |2200          |NULL          |UPI         |
|1008     

In [15]:
# 12
bookings_df.dropna(subset=['customer_name','service_type','booking_amount']).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007      |Imran Ali    |Pune     |Hotel       |Budget Inn      |2200          |NULL          |UPI         |
|1008     

In [16]:
# 13
bookings_df.fillna({'city': 'Unknown'}).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [17]:
# 14
bookings_df.fillna({'provider': 'Not Available'}).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |Not Available   |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [18]:
# 15
bookings_df.fillna({'payment_mode': 'Not Provided'}).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |Not Provided|
|1007     

In [19]:
# 16
bookings_df.fillna({'booking_status': 'Unknown'}).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [20]:
# 17
bookings_df.fillna({'booking_amount': 0}).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |0             |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [21]:
# 18
bookings_df.withColumn('data_quality_status',
    when(
        col('city').isNull() |
        col('provider').isNull() |
        col('booking_amount').isNull() |
        col('booking_status').isNull() |
        col('payment_mode').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |Complete           |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |Complete           |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |Incomplete         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |Incomplete         |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed

In [22]:
# 19
bookings_df.withColumn('data_quality_status',
    when(
        col('city').isNull() |
        col('provider').isNull() |
        col('booking_amount').isNull() |
        col('booking_status').isNull() |
        col('payment_mode').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).groupBy('data_quality_status').count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    7|
|         Incomplete|    8|
+-------------------+-----+



In [23]:
# 20
bookings_df.withColumn('data_quality_status',
    when(
        col('city').isNull() |
        col('provider').isNull() |
        col('booking_amount').isNull() |
        col('booking_status').isNull() |
        col('payment_mode').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).filter(col('data_quality_status') == 'Complete').show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |Complete           |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |Complete           |
|1008      |Meera Nair   |Kochi    |Hotel       |Hill View Resort|7500          |Confirmed     |Card        |Complete           |
|1009      |Rohan Das    |Kolkata  |Flight      |Air India       |7400          |Pending       |UPI         |Complete           |
|1010      |Nisha Reddy  |Bangalore|Flight      |British Airways |62000         |Confirmed

In [24]:
# 21
bookings_df.withColumn('data_quality_status',
    when(
        col('city').isNull() |
        col('provider').isNull() |
        col('booking_amount').isNull() |
        col('booking_status').isNull() |
        col('payment_mode').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).filter(col('data_quality_status') == 'Incomplete').show(truncate=False)

+----------+-------------+---------+------------+--------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|city     |service_type|provider      |booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+--------------+--------------+--------------+------------+-------------------+
|1003      |John Mathew  |NULL     |Flight      |Air India     |12000         |Confirmed     |UPI         |Incomplete         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL          |7500          |Pending       |Cash        |Incomplete         |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara       |NULL          |Confirmed     |Card        |Incomplete         |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo        |5900          |Cancelled     |NULL        |Incomplete         |
|1007      |Imran Ali    |Pune     |Hotel       |Budget Inn    |2200          |NULL          |UPI       

In [25]:
# 22
bookings_df.fillna({'booking_amount': 0}).withColumn('tax', col('booking_amount') * 0.05).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|tax   |
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |325.0 |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |225.0 |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |600.0 |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |375.0 |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |0             |Confirmed     |Card        |0.0   |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo         

In [26]:
# 23
bookings_df.fillna({'booking_amount': 0}) \
           .withColumn('tax', col('booking_amount') * 0.05) \
           .withColumn('final_amount', col('booking_amount') + col('booking_amount') * 0.05).show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|tax   |final_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |325.0 |6825.0      |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |225.0 |4725.0      |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |600.0 |12600.0     |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |375.0 |7875.0      |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |0             |Confirmed

In [27]:
# 24
bookings_df.filter(col('booking_status') == 'Confirmed') \
           .agg(sum('booking_amount').alias('confirmed_revenue')).show()

+-----------------+
|confirmed_revenue|
+-----------------+
|           147300|
+-----------------+



In [28]:
# 25
bookings_df.fillna({'service_type': 'Unknown'}) \
           .groupBy('service_type').count().show()

+------------+-----+
|service_type|count|
+------------+-----+
|     Unknown|    1|
|       Hotel|    6|
|      Flight|    8|
+------------+-----+



In [29]:
# 26
bookings_df.fillna({'city': 'Unknown'}) \
           .groupBy('city').count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|  Kolkata|    1|
|  Unknown|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [30]:
# 27
bookings_df.fillna({'booking_amount': 0}).agg(avg('booking_amount').alias('avg_with_zero')).show()

+------------------+
|     avg_with_zero|
+------------------+
|12353.333333333334|
+------------------+



In [31]:
# 28
bookings_df.dropna(subset=['booking_amount']).agg(avg('booking_amount').alias('avg_without_nulls')).show()

+------------------+
| avg_without_nulls|
+------------------+
|14253.846153846154|
+------------------+



In [32]:
# 29
avg_with_zero = bookings_df.fillna({'booking_amount': 0}).agg(avg('booking_amount')).collect()[0][0]
avg_without_nulls = bookings_df.dropna(subset=['booking_amount']).agg(avg('booking_amount')).collect()[0][0]
print(f"Avg with 0 for nulls  : {round(avg_with_zero, 2)}")
print(f"Avg after dropping nulls: {round(avg_without_nulls, 2)}")

Avg with 0 for nulls  : 12353.33
Avg after dropping nulls: 14253.85


In [33]:
# 30
clean_df = bookings_df.dropna()
clean_df.write.mode('overwrite').parquet(r'C:\new_env\hexaware-sql\june_10\clean_bookings.parquet')
print("Saved as clean_bookings.parquet")

Saved as clean_bookings.parquet


In [34]:
customers_df = spark.read.option("multiline","true").json(
    r"C:\new_env\hexaware-sql\june_10\customers.json"
)
customers_df.show(truncate=False)
customers_df.printSchema()

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: strin

In [35]:
flat_customers_df = customers_df.select(
    "customer_id", "name", "city", "membership",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
)
flat_customers_df.show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [36]:
# 1
customers_df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [37]:
# 2
customers_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)



In [38]:
# 3
customers_df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [39]:
# 4
flat_customers_df.select('customer_id','name','phone','email').show(truncate=False)

+-----------+------------+----------+---------------+
|customer_id|name        |phone     |email          |
+-----------+------------+----------+---------------+
|1          |Aarav Mehta |9876500011|aarav@mail.com |
|2          |Sana Khan   |NULL      |sana@mail.com  |
|3          |John Mathew |9876500013|NULL           |
|4          |Ayesha Begum|9876500014|ayesha@mail.com|
|5          |Vikram Rao  |NULL      |NULL           |
+-----------+------------+----------+---------------+



In [40]:
# 5
flat_customers_df.select('customer_id','name','preferred_service','budget_range').show(truncate=False)

+-----------+------------+-----------------+------------+
|customer_id|name        |preferred_service|budget_range|
+-----------+------------+-----------------+------------+
|1          |Aarav Mehta |Flight           |Medium      |
|2          |Sana Khan   |Hotel            |NULL        |
|3          |John Mathew |Flight           |High        |
|4          |Ayesha Begum|NULL             |Low         |
|5          |Vikram Rao  |Flight           |High        |
+-----------+------------+-----------------+------------+



In [41]:
# 6
flat_customers_df.select('name','city','phone','email').show(truncate=False)

+------------+---------+----------+---------------+
|name        |city     |phone     |email          |
+------------+---------+----------+---------------+
|Aarav Mehta |Hyderabad|9876500011|aarav@mail.com |
|Sana Khan   |Bangalore|NULL      |sana@mail.com  |
|John Mathew |NULL     |9876500013|NULL           |
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|Vikram Rao  |Mumbai   |NULL      |NULL           |
+------------+---------+----------+---------------+



In [42]:
# 7
flat_customers_df.filter(col('city').isNull()).show(truncate=False)

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|name       |city|membership|phone     |email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|3          |John Mathew|NULL|Gold      |9876500013|NULL |Flight           |High        |
+-----------+-----------+----+----------+----------+-----+-----------------+------------+



In [43]:
# 8
flat_customers_df.filter(col('phone').isNull()).show(truncate=False)

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|name      |city     |membership|phone|email        |preferred_service|budget_range|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|2          |Sana Khan |Bangalore|Silver    |NULL |sana@mail.com|Hotel            |NULL        |
|5          |Vikram Rao|Mumbai   |Platinum  |NULL |NULL         |Flight           |High        |
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+



In [44]:
# 9
flat_customers_df.filter(col('email').isNull()).show(truncate=False)

+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|customer_id|name       |city  |membership|phone     |email|preferred_service|budget_range|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|3          |John Mathew|NULL  |Gold      |9876500013|NULL |Flight           |High        |
|5          |Vikram Rao |Mumbai|Platinum  |NULL      |NULL |Flight           |High        |
+-----------+-----------+------+----------+----------+-----+-----------------+------------+



In [45]:
# 10
flat_customers_df.filter(col('membership').isNull()).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [46]:
# 11
flat_customers_df.filter(col('preferred_service').isNull()).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [47]:
# 12
flat_customers_df.filter(col('budget_range').isNull()).show(truncate=False)

+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|name     |city     |membership|phone|email        |preferred_service|budget_range|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|2          |Sana Khan|Bangalore|Silver    |NULL |sana@mail.com|Hotel            |NULL        |
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+



In [48]:
# 13
null_counts = flat_customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in flat_customers_df.columns
])
null_counts.show()

+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [49]:
# 14
flat_customers_df.fillna({'city': 'Unknown'}).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|3          |John Mathew |Unknown  |Gold      |9876500013|NULL           |Flight           |High        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [50]:
# 15
flat_customers_df.fillna({'membership': 'Standard'}).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |
|4          |Ayesha Begum|Hyderabad|Standard  |9876500014|ayesha@mail.com|NULL             |Low         |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [51]:
# 16
flat_customers_df.fillna({'phone': 'Not Provided'}).show(truncate=False)

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone       |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011  |aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan   |Bangalore|Silver    |Not Provided|sana@mail.com  |Hotel            |NULL        |
|3          |John Mathew |NULL     |Gold      |9876500013  |NULL           |Flight           |High        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014  |ayesha@mail.com|NULL             |Low         |
|5          |Vikram Rao  |Mumbai   |Platinum  |Not Provided|NULL           |Flight           |High        |
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [52]:
# 17
flat_customers_df.fillna({'email': 'Not Provided'}).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|3          |John Mathew |NULL     |Gold      |9876500013|Not Provided   |Flight           |High        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |Not Provided   |Flight           |High        |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [53]:
# 18
flat_customers_df.fillna({'preferred_service': 'Not Selected'}).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|Not Selected     |Low         |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [54]:
# 19
flat_customers_df.fillna({'budget_range': 'Unknown'}).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |Unknown     |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [55]:
# 20
flat_customers_df.withColumn('customer_quality_status',
    when(
        col('city').isNull() |
        col('phone').isNull() |
        col('email').isNull() |
        col('membership').isNull() |
        col('preferred_service').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |Complete               |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |Incomplete             |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |Incomplete             |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |Incomplete             |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight          

In [56]:
# 21
flat_customers_df.withColumn('customer_quality_status',
    when(
        col('city').isNull() |
        col('phone').isNull() |
        col('email').isNull() |
        col('membership').isNull() |
        col('preferred_service').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).groupBy('customer_quality_status').count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    1|
|             Incomplete|    4|
+-----------------------+-----+



In [57]:
# 22
flat_customers_df.withColumn('customer_quality_status',
    when(
        col('city').isNull() |
        col('phone').isNull() |
        col('email').isNull() |
        col('membership').isNull() |
        col('preferred_service').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).filter(col('customer_quality_status') == 'Complete').show(truncate=False)

+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|customer_id|name       |city     |membership|phone     |email         |preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|1          |Aarav Mehta|Hyderabad|Gold      |9876500011|aarav@mail.com|Flight           |Medium      |Complete               |
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+



In [58]:
# 23
flat_customers_df.withColumn('customer_quality_status',
    when(
        col('city').isNull() |
        col('phone').isNull() |
        col('email').isNull() |
        col('membership').isNull() |
        col('preferred_service').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).filter(col('customer_quality_status') == 'Incomplete').show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |Incomplete             |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |Incomplete             |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |Incomplete             |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |Incomplete             |
+-----------+------------+---------+----------+----------+---------------+----------------

In [59]:
# 24
flat_customers_df.fillna({'membership': 'Standard'}).groupBy('membership').count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [60]:
# 25
flat_customers_df.fillna({'preferred_service': 'Not Selected'}).groupBy('preferred_service').count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [61]:
# 26
flat_customers_df.write.mode('overwrite').parquet('customers_flat.parquet')
print("Saved as customers_flat.parquet")

Saved as customers_flat.parquet


In [62]:
# 27
clean_customers = flat_customers_df.dropna()
clean_customers.write.mode('overwrite').csv('clean_customers.csv', header=True)
print("Saved as clean_customers.csv")

Saved as clean_customers.csv


In [63]:
# 28
original_count = flat_customers_df.count()
clean_count = flat_customers_df.dropna().count()
print(f"Original count : {original_count}")
print(f"Clean count    : {clean_count}")
print(f"Dropped rows   : {original_count - clean_count}")

Original count : 5
Clean count    : 1
Dropped rows   : 4


In [64]:
# 29
flat_customers_df.filter(
    col('phone').isNull() | col('email').isNull()
).show(truncate=False)

+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|name       |city     |membership|phone     |email        |preferred_service|budget_range|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|2          |Sana Khan  |Bangalore|Silver    |NULL      |sana@mail.com|Hotel            |NULL        |
|3          |John Mathew|NULL     |Gold      |9876500013|NULL         |Flight           |High        |
|5          |Vikram Rao |Mumbai   |Platinum  |NULL      |NULL         |Flight           |High        |
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+



In [65]:
# 30
flat_customers_df.filter(
    col('preferred_service').isNull() | col('budget_range').isNull()
).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+

